[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Error estándar e intervalos de confianza.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se calcularán el error estándar e intervalos de confianza para medias y proporciones, y se revisará cuándo se puede confiar en la fórmula *t*.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

* La generación de números aleatorios con `np.random.default_rng` se explica en [py311 · 04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) y los resúmenes con *NumPy* en [py311 · 07 Gestión y análisis de datos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/07_gestion_y_analisis_de_datos_de_numpy.ipynb).
* py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html), [*SciPy* · `bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) y [*statsmodels* · `proportion_confint`](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportion_confint.html).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluidos `stats` de *SciPy* y las funciones de *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.weightstats import DescrStatsW

**¿Por qué este paso?** Se parte del archivo limpio de la sesión 2 para no repetir la limpieza; si no existe (por ejemplo, en un *Codespace* nuevo), se regenera con los mismos pasos mínimos. `titanic` se usará para proporciones y para un ejemplo de datos muy asimétricos.

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará desde `penguins.csv` si no existe) y `titanic.csv`.

In [2]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
pinguinos.shape, titanic.shape

((342, 7), (891, 15))

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `poblacion` (la masa de todos los pingüinos, usada como población de juguete) y `tarifas` (las tarifas del Titanic), que se usaron en `16_distribucion_muestral.ipynb`.

In [3]:
poblacion = pinguinos["body_mass_g"].dropna().to_numpy()
tarifas = titanic["fare"].to_numpy()
len(poblacion), len(tarifas)

(342, 891)

---
## 1. Error estándar e intervalos de confianza (25 min).

En la práctica solo hay **una** muestra. El error estándar se estima con la desviación estándar muestral: EE = *s* / √*n*. Un **intervalo de confianza** (IC) del 95% para la media es:

```
x̄ ± t(0.975, n − 1) · s / √n
```

Donde:
* `t(0.975, n − 1)` es el cuantil de la distribución *t* de Student con *n* − 1 grados de libertad (≈ 1.98 para *n* = 123).
* `s / √n` es el error estándar estimado.

**Ejemplo:** desde aquí se vuelve a la situación real: los Gentoo medidos son una **muestra** de la población de Gentoo.

**Concepto: de la distribución muestral al intervalo.** Por el TCL, $\bar{x}$ cae a menos de $1.96\,EE$ de $\mu$ en 95% de las muestras. Si eso es cierto, también es cierto que $\mu$ está a menos de $1.96\,EE$ de $\bar{x}$; de ahí sale el intervalo. Como $\sigma$ es desconocida y se estima con $s$, se usa el cuantil de la *t* de Student, un poco mayor que 1.96, para compensar la incertidumbre extra de estimar $\sigma$:

$$IC_{95\%} = \bar{x} \pm t_{0.975,\,n-1}\,\frac{s}{\sqrt{n}}$$

* La siguiente celda calculará el IC del 95% para la masa media de los Gentoo con la fórmula, paso a paso.

In [4]:
masa_gentoo = pinguinos.loc[pinguinos["species"] == "Gentoo", "body_mass_g"].dropna()
n = len(masa_gentoo)
media = masa_gentoo.mean()
error_estandar = masa_gentoo.std(ddof=1) / np.sqrt(n)
t_critico = stats.t.ppf(0.975, df=n - 1)
n, round(float(media), 1), round(float(error_estandar), 1), np.round([media - t_critico * error_estandar, media + t_critico * error_estandar], 1)

(123, 5076.0, 45.5, array([4986., 5166.]))

**Interpretación:** con 123 Gentoo, la masa media es 5,076 g con EE de 45.5 g; el IC del 95% va de 4,986 a 5,166 g. El margen (≈ 90 g) es dos errores estándar. Ese margen es mucho menor que la dispersión de los individuos (≈ 500 g), porque describe la incertidumbre sobre **la media**, no sobre cada pingüino.

* La siguiente celda obtendrá el mismo intervalo con `stats.t.interval` de *SciPy* y con `DescrStatsW` de *statsmodels*.

In [5]:
con_scipy = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(masa_gentoo))
con_statsmodels = DescrStatsW(masa_gentoo).tconfint_mean(alpha=0.05)
np.round(con_scipy, 1), np.round(con_statsmodels, 1)

(array([4986., 5166.]), array([4986., 5166.]))

**Interpretación:** las dos bibliotecas reproducen exactamente el cálculo a mano. En un informe se usa una de ellas (menos riesgo de errores de escritura), pero conviene saber qué fórmula aplican. `stats.sem` usa `ddof=1` por defecto.

### 1.1 Intervalos para una proporción.

Para una proporción *p̂* = *k* / *n* hay varios métodos (`proportion_confint(k, n, method=...)`):

| Método | `method=` | Comentario |
|---|---|---|
| Wald | `"normal"` | *p̂* ± 1.96·√(*p̂*(1 − *p̂*)/*n*). Simple, pero falla con *n* pequeño o *p̂* cerca de 0 o 1 |
| Wilson | `"wilson"` | Recomendado por defecto: buena cobertura incluso con *n* pequeño |
| Clopper-Pearson | `"beta"` | "Exacto": conservador (intervalos algo más anchos) |

**Concepto:** una proporción es la media de una variable 0/1, así que su error estándar es $EE(\hat{p}) = \sqrt{\hat{p}(1-\hat{p})/n}$. Wald usa directamente $\hat{p} \pm 1.96\,EE$; Wilson "invierte" la prueba *z* y produce un intervalo que nunca sale de [0, 1] y que se centra un poco hacia 0.5; Clopper-Pearson usa la distribución binomial exacta.

* La siguiente celda calculará los tres intervalos para la tasa de supervivencia del Titanic.

In [6]:
sobrevivientes, pasajeros = int(titanic["survived"].sum()), len(titanic)
pd.DataFrame({metodo: proportion_confint(sobrevivientes, pasajeros, alpha=0.05, method=metodo)
              for metodo in ["normal", "wilson", "beta"]}, index=["inferior", "superior"]).T.round(3)

,inferior,superior
normal,0.352,0.416
wilson,0.352,0.416
beta,0.352,0.417


Con 891 pasajeros los tres métodos coinciden (≈ 35% a 42%). La decisión importa con muestras pequeñas.

**Decisión: ¿Wald o Wilson con pocos datos?** Con 2 éxitos en 30 casos, la fórmula de Wald da un límite inferior **negativo**, imposible para una proporción (`statsmodels` lo recorta a 0, lo que oculta el problema).

* La siguiente celda calculará Wald a mano y Wilson con *statsmodels* para 2 éxitos en 30.

In [7]:
p_gorro = 2 / 30
margen = 1.96 * np.sqrt(p_gorro * (1 - p_gorro) / 30)
{"Wald a mano": (round(float(p_gorro - margen), 3), round(float(p_gorro + margen), 3)),
 "Wilson": tuple(round(float(v), 3) for v in proportion_confint(2, 30, method="wilson"))}

{'Wald a mano': (-0.023, 0.156), 'Wilson': (0.018, 0.213)}

**Interpretación:** Wald da (−0.023, 0.156), con un límite inferior imposible; Wilson da (0.018, 0.213), asimétrico alrededor de $\hat{p}$ = 0.067 como corresponde a una proporción cercana a 0. **Decisión:** usar Wilson por defecto, sobre todo si $n\hat{p}$ o $n(1-\hat{p})$ es menor que 10.

### 1.2 Consecuencia: ¿cuándo se puede confiar en la fórmula *t*?

El "95%" es una promesa sobre el **procedimiento**: si se repitiera el muestreo, 95% de los intervalos capturarían μ. Esa promesa depende de supuestos. Se simulará la **cobertura real** del IC *t* con dos poblaciones: la masa de los pingüinos (sin colas largas) y la tarifa del Titanic (muy asimétrica, con atípicos).

**Concepto: cobertura.** Es la proporción de intervalos, sobre muchas muestras repetidas, que contienen el parámetro verdadero. Un método "del 95%" que en realidad cubre 80% es engañoso. Solo se puede medir por simulación, cuando se conoce la población.

* La siguiente celda definirá `cobertura_t`, que saca muchas muestras, calcula el IC *t* de cada una y cuenta qué fracción contiene la media poblacional.

In [8]:
def cobertura_t(poblacion, n, repeticiones=2000, generador=rng):
    muestras = generador.choice(poblacion, size=(repeticiones, n))
    medias = muestras.mean(axis=1)
    errores = muestras.std(axis=1, ddof=1) / np.sqrt(n)
    t_critico = stats.t.ppf(0.975, df=n - 1)
    contiene = (medias - t_critico * errores <= poblacion.mean()) & (poblacion.mean() <= medias + t_critico * errores)
    return contiene.mean()

* La siguiente celda calculará la cobertura con muestras de 10, 30 y 100 en ambas poblaciones (`tarifas` se definió en la sección 2.2 de `16_distribucion_muestral.ipynb`).

In [9]:
pd.DataFrame({n: {"masa (pingüinos)": cobertura_t(poblacion, n), "tarifa (Titanic)": cobertura_t(tarifas, n)}
              for n in [10, 30, 100]}).round(3)

,10,30,100
masa (pingüinos),0.946,0.943,0.953
tarifa (Titanic),0.798,0.867,0.904


Con la masa, la cobertura es ≈ 95% incluso con *n* = 10. Con la tarifa, un IC "del 95%" con *n* = 10 solo captura la media en torno a 80% de las veces, y aún con *n* = 100 se queda debajo de 95%. **Consecuencia:** con datos muy asimétricos y muestras pequeñas, el intervalo es demasiado estrecho y transmite una seguridad falsa. Conviene reportar la mediana con un IC *bootstrap*, transformar (logaritmo) o, al menos, advertirlo.

**💡 Ideas clave de la sección 1:**
* $IC_{95\%} = \bar{x} \pm t\,s/\sqrt{n}$ para la media; Wilson para proporciones.
* El 95% es una propiedad del **procedimiento**, que solo se cumple si se cumplen sus supuestos.
* Con datos muy asimétricos y $n$ pequeño, el IC *t* cubre menos de lo prometido.

### 🟢 Ejercicio 1 · Supervivencia en tercera clase.

Calcula la tasa de supervivencia de los pasajeros de **tercera clase** con su IC del 95% por Wilson, y compárala con la de primera clase. ¿Se traslapan los intervalos?

**🎯 Objetivo:** calcular IC de proporciones por grupo con `proportion_confint`. **Qué se espera aprender:** a comparar grupos con su incertidumbre, no solo con sus tasas puntuales.

* La siguiente celda muestra una solución posible.

In [10]:
por_clase = titanic.groupby("class")["survived"].agg(sobrevivientes="sum", pasajeros="count")
intervalos = [proportion_confint(f["sobrevivientes"], f["pasajeros"], method="wilson") for _, f in por_clase.iterrows()]
por_clase.assign(
    tasa=lambda d: d["sobrevivientes"] / d["pasajeros"],
    ic_inferior=[i[0] for i in intervalos],
    ic_superior=[i[1] for i in intervalos],
).round(3)

,sobrevivientes,pasajeros,tasa,ic_inferior,ic_superior
class,,,,,
First,136,216,0.630,0.563,0.691
Second,87,184,0.473,0.402,0.545
Third,119,491,0.242,0.207,0.282


**✅ Respuesta sugerida:**

En tercera clase sobrevivió 24.2% (IC 95% de Wilson ≈ 20.7% a 28.2%, con 491 pasajeros); en primera, 63.0% (≈ 56.4% a 69.1%). Los intervalos están muy lejos uno de otro: la diferencia no se explica por variación muestral.

### 🔴 Reto 1 · Cobertura de Wald y Wilson.

Simula 5,000 muestras de tamaño 20 de una población con proporción verdadera 0.10 (`generador.binomial(20, 0.10, size=5000)`, con `generador = np.random.default_rng(401)`). Para cada muestra calcula el IC de Wald y el de Wilson y estima la cobertura de cada método. ¿Cuál cumple mejor la promesa del 95%?

**🎯 Objetivo:** medir por simulación la cobertura real de dos métodos. **Qué se espera aprender:** a verificar una afirmación metodológica ("Wilson es mejor") con datos simulados en lugar de aceptarla por autoridad.

* La siguiente celda muestra una solución posible.

In [11]:
generador = np.random.default_rng(401)
exitos = generador.binomial(20, 0.10, size=5000)
cobertura = {}
for metodo in ["normal", "wilson"]:
    inferior, superior = proportion_confint(exitos, 20, method=metodo)
    cobertura[metodo] = float(((inferior <= 0.10) & (0.10 <= superior)).mean())
cobertura

{'normal': 0.8734, 'wilson': 0.9544}

**✅ Respuesta sugerida:**

Wald cubre bastante menos que 95% (cerca de 87%): cuando la muestra tiene 0 éxitos su intervalo es [0, 0] y nunca contiene 0.10. Wilson queda mucho más cerca de 95%. Por eso Wilson es la opción por defecto para proporciones.

---
## ✅ Cierre.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `18_bootstrap_e_interpretacion_de_intervalos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>